# 🧠 EpiConnectome Tutorial
## A Step-by-Step Guide to EEG Connectivity Analysis of Epileptic Seizures

**Project:** EpiConnectome — Brainhack School 2026  
**Dataset:** Siena Scalp EEG Database (PhysioNet)  
**GitHub:** https://github.com/duhmariya/EEG_Project

---

### What you will learn in this notebook

1. How to load seizure EEG data from the Siena dataset
2. How to preprocess EEG signals (filtering, bad channel detection, ICA)
3. How to compute **wPLI** functional connectivity across frequency bands
4. How to visualize connectivity as circle plots and bar charts
5. How to compute graph theory metrics from connectivity matrices
6. How to save results to Excel for further analysis

### Prerequisites

- Basic Python knowledge
- NumPy/pandas familiarity
- No prior EEG experience required!

### Setup
```bash
conda env create -f environment.yml
conda activate epiconnectome
jupyter notebook
```

---
## 📦 Step 0 — Import Libraries

In [ ]:
import os
import mne
import mne_connectivity
from mne_connectivity import spectral_connectivity_epochs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from mne_connectivity.viz import plot_connectivity_circle

# Suppress MNE verbose output for cleaner notebook
mne.set_log_level('WARNING')

# Reproducibility — ALWAYS set this first
import random
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
mne.set_config('MNE_RANDOM_SEED', str(RANDOM_SEED))
mne.set_config('MNE_USE_NUMBA', 'false')

print('✅ All libraries imported successfully')
print(f'   MNE version: {mne.__version__}')
print(f'   MNE-connectivity version: {mne_connectivity.__version__}')

---
## 📂 Step 1 — Load the Data

### About the Siena Dataset

The **Siena Scalp EEG Database** contains EEG recordings from 14 epilepsy patients captured at the University of Siena. Each file covers one seizure event and the surrounding recording.

| Property | Value |
|---|---|
| Patients | 14 subjects |
| Seizures | 47 seizures |
| Sampling rate | 512 Hz |
| Channels | 29 scalp electrodes |
| Format | EDF (converted to TXT by our pipeline) |

### Why TXT format?

Our `01_siena_to_txt_converter.py` script extracts the seizure segments from the original EDF files and saves them as plain text matrices (samples × channels). This makes them easier to load and share.

> 📥 **Download the dataset:** `wget -r -N -c -np https://physionet.org/files/siena-scalp-eeg/1.0.0/`

In [ ]:
# ── CHANGE THIS PATH to your TXT file ──────────────────────────
TXT_FILE_PATH = 'path/to/your/seizure_file.txt'  # e.g. 'data/PN00_seiz1.txt'
# ───────────────────────────────────────────────────────────────

# Load the TXT file
# The file is a 2D matrix: rows = time samples, columns = EEG channels
try:
    eeg_data = np.loadtxt(TXT_FILE_PATH)
except:
    eeg_data = pd.read_csv(TXT_FILE_PATH, sep='\s+', header=None).values

# Trim first/last 30 seconds (often noisy at recording boundaries)
SFREQ = 512  # Original Siena sampling rate
trim_samples = 30 * SFREQ
if eeg_data.shape[0] > 2 * trim_samples:
    eeg_data = eeg_data[trim_samples:-trim_samples, :]

# Use first 16 channels (standard 10-20 system)
eeg_data = eeg_data[:, :16]

print(f'✅ Data loaded: {eeg_data.shape[0]} samples × {eeg_data.shape[1]} channels')
print(f'   Duration: {eeg_data.shape[0]/SFREQ:.1f} seconds')

---
## 🔧 Step 2 — Create MNE Raw Object

### What is MNE-Python?

**MNE-Python** is the standard open-source library for EEG/MEG analysis. It provides tools for loading, preprocessing, and analyzing brain signals.

We need to tell MNE:
- **Channel names** — which electrode is which
- **Channel types** — all EEG here
- **Sampling frequency** — 512 Hz
- **Montage** — where the electrodes are physically located on the scalp

The **standard_1020 montage** defines the 3D coordinates of electrodes in the standard 10-20 system.

In [ ]:
# Standard 16 channel names (10-20 system)
CHANNEL_NAMES = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'T3', 'C3', 'Cz', 'C4', 'T4',
    'T5', 'P3', 'Pz', 'P4'
]

# Create MNE info object
info = mne.create_info(
    ch_names=CHANNEL_NAMES,
    sfreq=SFREQ,
    ch_types=['eeg'] * 16
)

# Create Raw object — note: MNE expects (channels × samples), so we transpose
raw = mne.io.RawArray(eeg_data.T, info, verbose=False)

# Set electrode locations using standard 10-20 montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, match_alias=True, on_missing='ignore')

print(f'✅ MNE Raw object created')
print(f'   Channels: {len(raw.ch_names)}')
print(f'   Duration: {raw.times[-1]:.1f} s')
print(f'   Sampling rate: {raw.info["sfreq"]} Hz')

# Quick preview of the raw signal
raw.plot(duration=5, n_channels=8, scalings='auto', title='Raw EEG (first 5 seconds)');

---
## 🧹 Step 3 — Preprocessing

Raw EEG contains many artifacts. We need to clean the signal before computing connectivity. Our pipeline has 4 preprocessing steps:

### 3a. Bandpass Filtering (4–40 Hz)

We keep only the frequency range of interest:
- **Low cutoff (4 Hz):** removes slow drift and movement artifacts
- **High cutoff (40 Hz):** removes high-frequency muscle noise

### 3b. Resampling (512 → 1000 Hz)

We upsample to 1000 Hz for consistency with other datasets and better temporal resolution.

### 3c. Bad Channel Detection (LOF)

**Local Outlier Factor (LOF)** finds channels that are statistical outliers compared to their neighbors — these are likely broken electrodes or noisy leads.

### 3d. ICA Artifact Removal

**Independent Component Analysis (ICA)** separates brain signals from:
- **EOG artifacts** — eye blinks and movements (detected via Fp1, Fp2)
- **Muscle artifacts** — high-frequency EMG noise

In [ ]:
# ── 3a. Bandpass filter ──────────────────────────────────────────
print('Step 3a: Filtering 4–40 Hz...')
raw.filter(l_freq=4, h_freq=40, verbose=False)
print('   ✅ Done')

# ── 3b. Resample ─────────────────────────────────────────────────
print('Step 3b: Resampling to 1000 Hz...')
raw.resample(1000, verbose=False)
TARGET_SFREQ = 1000
print(f'   ✅ New sampling rate: {raw.info["sfreq"]} Hz')

# ── 3c. Common average reference ─────────────────────────────────
raw, _ = mne.set_eeg_reference(raw, projection=True, verbose=False)
print('   ✅ Reference set')

# ── 3d. Bad channel detection ────────────────────────────────────
print('Step 3c: Detecting bad channels (LOF)...')
bad_channels, scores = mne.preprocessing.find_bad_channels_lof(
    raw, n_neighbors=8, threshold=1.5, return_scores=True
)
print(f'   Bad channels found: {bad_channels if bad_channels else "None"}')

raw.info['bads'] = bad_channels
if bad_channels:
    raw.interpolate_bads(reset_bads=True)
    print(f'   ✅ Interpolated {len(bad_channels)} bad channels')

# ── 3e. ICA ──────────────────────────────────────────────────────
print('Step 3d: Running ICA (this may take ~1 minute)...')

ica = mne.preprocessing.ICA(
    n_components=15,
    random_state=RANDOM_SEED,
    method='infomax',
    max_iter='auto'
)

ica.fit(inst=raw.copy().filter(4, 40, verbose=False), verbose=False)

# Detect eye movement artifacts using frontal channels
eog_channels = [ch for ch in ['Fp1', 'Fp2'] if ch in raw.ch_names]
if eog_channels:
    eog_indices, _ = ica.find_bads_eog(
        inst=raw, ch_name=eog_channels,
        measure='correlation', threshold=0.5
    )
else:
    eog_indices = []

# Detect muscle artifacts
muscle_indices, _ = ica.find_bads_muscle(inst=raw, threshold=0.5)

ica.exclude = list(set(eog_indices + muscle_indices))
raw = ica.apply(raw.copy())

print(f'   ✅ Excluded {len(ica.exclude)} ICA components')
print(f'      EOG components: {eog_indices}')
print(f'      Muscle components: {muscle_indices}')
print('\n✅ Preprocessing complete!')

---
## ✂️ Step 4 — Epoching

### What is an Epoch?

An **epoch** is a short, fixed-length segment of continuous EEG. We cut the long recording into non-overlapping 10-second windows.

Why 10 seconds?
- Long enough to capture oscillatory dynamics in theta/alpha/beta bands
- Short enough to assume signal stationarity within each window
- Gives us multiple estimates of connectivity to average over

Connectivity will be computed **across all epochs** and averaged — this gives a more stable estimate than using the whole recording at once.

In [ ]:
EPOCH_LENGTH = 10  # seconds

# Create evenly spaced events every EPOCH_LENGTH seconds
times = raw.n_times
interval = TARGET_SFREQ * EPOCH_LENGTH
events = np.array([[i, 0, 1] for i in range(0, times, interval)])

# Create Epochs object
epochs = mne.Epochs(
    raw, events,
    event_id=1,
    tmin=0,
    tmax=EPOCH_LENGTH - 1/TARGET_SFREQ,
    baseline=None,
    preload=True,
    verbose=False
)

print(f'✅ Epochs created')
print(f'   Number of epochs: {len(epochs)}')
print(f'   Epoch length: {EPOCH_LENGTH} seconds')
print(f'   Samples per epoch: {len(epochs.times)}')
print(f'   Total data used: {len(epochs) * EPOCH_LENGTH} seconds')

---
## 🔗 Step 5 — Compute wPLI Connectivity

### What is wPLI?

**Weighted Phase Lag Index (wPLI)** measures the consistency of phase relationships between two EEG channels.

- **Value range:** 0 to 1
  - `0` = no consistent phase relationship (no connectivity)
  - `1` = perfectly consistent phase relationship (strong connectivity)

### Why wPLI instead of correlation or coherence?

| Metric | Robust to volume conduction? | Commonly used? |
|---|---|---|
| Pearson correlation | ❌ No | Yes |
| Coherence | ❌ No | Yes |
| PLI | ✅ Yes | Yes |
| **wPLI** | ✅ **Yes** | **Yes** |

**Volume conduction** is a major artifact in scalp EEG where the same brain source appears in multiple electrodes due to electrical spread through the skull. wPLI specifically discards zero-lag interactions, which are characteristic of volume conduction.

### Frequency Bands

| Band | Range | Brain function |
|---|---|---|
| **Theta** | 6–8 Hz | Memory, hippocampal activity |
| **Alpha** | 8–12 Hz | Relaxation, inhibition |
| **Beta** | 12–30 Hz | Active cognition, motor |

We compute wPLI separately for each band to see which frequency range shows the strongest connectivity during seizures.

In [ ]:
# Define frequency bands
FREQ_BANDS = {
    'theta': (6, 8),
    'alpha': (8, 12),
    'beta':  (12, 30)
}

# Extract fmin and fmax lists for MNE
fmin = [band[0] for band in FREQ_BANDS.values()]
fmax = [band[1] for band in FREQ_BANDS.values()]

print('Computing wPLI connectivity for all frequency bands...')
print(f'   Bands: {FREQ_BANDS}')

# Compute spectral connectivity across epochs
# - method='wpli': Weighted Phase Lag Index
# - mode='multitaper': uses multiple tapers for better spectral estimation
# - faverage=True: average within each frequency band
con = mne_connectivity.spectral_connectivity_epochs(
    epochs,
    method='wpli',
    mode='multitaper',
    fmin=fmin,
    fmax=fmax,
    faverage=True,
    sfreq=epochs.info['sfreq'],
    verbose=False
)

# Extract connectivity matrices for each band
con_matrix_raw = con.get_data(output='dense')  # shape: (channels, channels, n_bands)

conn_dict = {}
for idx, band_name in enumerate(FREQ_BANDS.keys()):
    matrix = con_matrix_raw[:, :, idx]
    # Symmetrize: MNE returns upper triangle only
    matrix = matrix + matrix.T
    np.fill_diagonal(matrix, 0)
    conn_dict[band_name] = matrix

print('\n✅ wPLI connectivity computed!')
for band, matrix in conn_dict.items():
    mean_conn = np.mean(matrix[matrix != 0])
    print(f'   {band.upper():6s}: mean wPLI = {mean_conn:.3f}')

---
## 📊 Step 6 — Visualize: Connectivity Circle Plot

The **circle plot (chord diagram)** shows all pairwise connections between channels.

- Each segment around the circle = one EEG channel
- Lines inside = connections between pairs
- **Color** = connection strength (dark = weak, yellow = strong)
- **Thickness** = connection strength

This gives an immediate visual impression of the overall connectivity pattern — which channels are most strongly connected during the seizure.

In [ ]:
ch_names = raw.ch_names
VISUALIZATION_THRESHOLD = 0.0

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
band_colors = {'theta': 'hot', 'alpha': 'hot', 'beta': 'hot'}

for ax, (band_name, con_matrix) in zip(axes, conn_dict.items()):
    # Build connectivity array for MNE circle plot
    # MNE expects upper triangle in specific order
    n_ch = len(ch_names)
    upper_tri = con_matrix[np.triu_indices(n_ch, k=1)]
    
    fig_circle, ax_circle = plot_connectivity_circle(
        upper_tri,
        ch_names,
        n_lines=None,
        vmin=VISUALIZATION_THRESHOLD,
        vmax=con_matrix.max(),
        title=f'{band_name.capitalize()} Band',
        colormap='hot',
        colorbar=True,
        show=False
    )
    plt.savefig(f'circle_{band_name.capitalize()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'   ✅ Saved: circle_{band_name.capitalize()}.png')

---
## 📊 Step 7 — Visualize: Channel Bar Chart

The **bar chart** shows the total connectivity strength of each channel — how connected each electrode is to all others.

This is computed as the **sum of wPLI values** for each channel across all its pairs.

Channels with tall bars are "hubs" in the seizure network — they have strong phase synchrony with many other regions.

In [ ]:
BAR_THRESHOLD = 0.3

fig, axes = plt.subplots(1, 3, figsize=(24, 5))
colors = {'theta': '#4ECDC4', 'alpha': '#45B7D1', 'beta': '#96CEB4'}

for ax, (band_name, con_matrix) in zip(axes, conn_dict.items()):
    # Channel strength = sum of connections for each channel
    channel_strength = np.mean(con_matrix, axis=1)

    bars = ax.bar(ch_names, channel_strength,
                  color=colors[band_name], alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axhline(y=BAR_THRESHOLD, color='r', linestyle='--',
               linewidth=2, label=f'Threshold = {BAR_THRESHOLD}')

    ax.set_title(f'{band_name.capitalize()} Band ({FREQ_BANDS[band_name][0]}–{FREQ_BANDS[band_name][1]} Hz)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Channels', fontsize=11)
    ax.set_ylabel('Mean wPLI Strength', fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Channel Connectivity Strength by Frequency Band', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('channel_connectivity_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: channel_connectivity_bars.png')

---
## 🕸️ Step 8 — Graph Theory Metrics

We can model the connectivity matrix as a **graph** (network):
- **Nodes** = EEG channels
- **Edges** = connections above a threshold
- **Edge weights** = wPLI values

Graph theory gives us summary statistics:

| Metric | What it means |
|---|---|
| **Global Efficiency** | How efficiently information can travel across the network |
| **Clustering Coefficient** | How much channels tend to form local clusters |
| **Density** | Fraction of possible connections that are active |
| **Modularity** | Whether the network separates into distinct sub-networks |
| **Node Strength** | Total connectivity weight per channel |

In [ ]:
THRESHOLD = 0.5  # Only keep connections above this value

results = {}

for band_name, con_matrix in conn_dict.items():
    G = nx.Graph()
    n_nodes = con_matrix.shape[0]

    # Add edges above threshold
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if con_matrix[i, j] > THRESHOLD:
                G.add_edge(i, j, weight=con_matrix[i, j])

    if G.number_of_edges() > 0:
        metrics = {
            'n_edges':          G.number_of_edges(),
            'density':          nx.density(G),
            'global_efficiency': nx.global_efficiency(G),
            'avg_clustering':   nx.average_clustering(G, weight='weight'),
        }
        try:
            communities = nx.community.greedy_modularity_communities(G)
            metrics['modularity'] = nx.community.modularity(G, communities)
            metrics['n_communities'] = len(communities)
        except:
            metrics['modularity'] = float('nan')
            metrics['n_communities'] = float('nan')
    else:
        metrics = {'n_edges': 0, 'density': 0, 'global_efficiency': 0,
                   'avg_clustering': 0, 'modularity': float('nan'), 'n_communities': float('nan')}

    results[band_name] = metrics

# Display results as a table
df_results = pd.DataFrame(results).T
df_results.index.name = 'Band'
print('\n📊 Graph Theory Metrics:')
print(df_results.round(4).to_string())

---
## 💾 Step 9 — Save Results to Excel

We save the connectivity matrices to Excel with one sheet per frequency band. This creates a **connectivity atlas entry** for this seizure.

Once you run this pipeline on all 47 seizures in the Siena dataset, you'll have a complete connectivity atlas — 47 files × 3 bands × 16×16 matrices.

In [ ]:
output_file = 'connectivity_results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for band_name, con_matrix in conn_dict.items():
        df = pd.DataFrame(
            con_matrix,
            index=ch_names,
            columns=ch_names
        )
        df.to_excel(writer, sheet_name=band_name.capitalize())

    # Also save graph metrics
    df_results.to_excel(writer, sheet_name='Graph_Metrics')

print(f'✅ Results saved to: {output_file}')
print(f'   Sheets: Theta, Alpha, Beta, Graph_Metrics')
print(f'   Matrix size: {len(ch_names)} × {len(ch_names)} per band')

---
## 🔍 Step 10 — Connectivity Heatmap

A final visualization — the full connectivity matrix as a heatmap. This shows which channel pairs have the strongest wPLI values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
cmaps = {'theta': 'YlOrRd', 'alpha': 'YlGnBu', 'beta': 'PuRd'}

for ax, (band_name, con_matrix) in zip(axes, conn_dict.items()):
    sns.heatmap(
        con_matrix,
        xticklabels=ch_names,
        yticklabels=ch_names,
        cmap=cmaps[band_name],
        vmin=0, vmax=1,
        annot=False,
        square=True,
        cbar_kws={'label': 'wPLI'},
        ax=ax
    )
    ax.set_title(f'{band_name.capitalize()} Band\n({FREQ_BANDS[band_name][0]}–{FREQ_BANDS[band_name][1]} Hz)',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='both', labelsize=8)

plt.suptitle('wPLI Connectivity Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('connectivity_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: connectivity_heatmaps.png')

---
## ✅ Summary

You've completed the full EpiConnectome pipeline! Here's what you did:

| Step | What you did |
|---|---|
| 1 | Loaded seizure EEG from Siena TXT file |
| 2 | Created MNE Raw object with electrode locations |
| 3 | Preprocessed: filter → resample → bad channels → ICA |
| 4 | Cut recording into 10-second epochs |
| 5 | Computed wPLI connectivity for θ, α, β bands |
| 6 | Visualized as connectivity circle plots |
| 7 | Visualized as channel strength bar charts |
| 8 | Computed graph theory metrics |
| 9 | Saved 16×16 matrices to Excel |
| 10 | Visualized full matrix as heatmap |

---

### 🚀 Next Steps

- Run `02_channelbasedcode_Siena.py` to process **all 47 seizures** automatically
- Run `03_siena_feature_extraction.py` to extract entropy/variance features
- Run `04_siencehisto.py` to batch-visualize all connectivity Excel files
- Coming soon: Source-level analysis with dSPM + HCPMMP1
- Coming soon: GNN/GATv2 classification using connectivity matrices

---

### 📚 References

- Detti, P. (2020). Siena Scalp EEG Database. PhysioNet. https://doi.org/10.13026/5d4a-j060
- Gramfort et al. (2013). MEG and EEG data analysis with MNE-Python. Frontiers in Neuroscience.
- Vinck et al. (2011). An improved index of phase-synchronization for electrophysiological data. NeuroImage.

---
*EpiConnectome · Brainhack School 2026 · github.com/duhmariya/EEG_Project*